# 🔬 Section 4 Project: Multi-Agent Research System

This capstone notebook combines every Multi-Agent pattern from Section 4 into a single, working research pipeline: a **supervisor** plans the work, multiple **search agents** run **in parallel** via the `Send` API, an **analyst** and **report writer** synthesize a shared **blackboard** of findings, and a **quality checker** drives an **iterative refinement loop** until the report is good enough to ship.

## Learning Objectives
In this notebook, you will learn:
1. **Supervisor Architecture** - how a planning node can decompose a broad topic into targeted, parallelizable sub-tasks
2. **Parallel Execution with `Send`** - how to dynamically fan out work to multiple agent instances and fan the results back in
3. **Shared State / Blackboard Pattern** - how independent agents can accumulate results into one shared piece of state (`findings`)
4. **Iterative Refinement Loops** - how a quality-gate conditional edge can send work back for revision until it meets a bar, with a safety cap on iterations
5. **End-to-End Graph Composition** - how to wire all of the above nodes and edges into one compiled `StateGraph`

## Prerequisites
- Section 4.3 (Supervisor architecture), 4.5 (Parallel execution / `Send` API), and 4.6 (Shared state / blackboard)
- Comfort with LangGraph `StateGraph`, conditional edges, and structured output (Pydantic)
- An `OPENAI_API_KEY` available via a `.env` file (loaded with `python-dotenv`)

> Converted from `07_multi_agent_research_system.py` — part of **04 Multi Agent Systems**.

---
## 📦 Part 1: Environment Setup

We load environment variables from `.env` and create two `ChatOpenAI` instances: a deterministic `llm` (temperature `0`) for planning/search/analysis/scoring, and a `creative_llm` (temperature `0.7`) for drafting the report text itself.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LLM Initialization
# ============================================================================
import json
import operator
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send

load_dotenv()

# Deterministic LLM for planning, search, analysis, and quality scoring
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Higher-temperature LLM for drafting the final report (more natural prose)
creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print(f"🤖 LLM initialized: {llm.model_name} (temperature=0)")
print(f"🤖 Creative LLM initialized: {creative_llm.model_name} (temperature=0.7)")
print("✅ Environment ready!")

---
## 🧠 Part 2: State Schemas

Two `TypedDict` schemas drive this graph. `ResearchState` is the main graph state — it carries the shared **blackboard** (`findings`) that every search agent writes into via `operator.add`. `SearchTaskState` is the smaller, per-task state that each parallel `search_agent` instance receives from the `Send` API.

### Key Concepts:
- **`Annotated[..., operator.add]`**: tells LangGraph to *append* new findings to the existing list rather than overwrite it — this is what makes the blackboard pattern work across parallel branches
- **`Annotated[..., add_messages]`**: the standard LangGraph reducer for accumulating a conversation/activity trace

In [ ]:
# ============================================================================
# STATE SCHEMAS: ResearchState and SearchTaskState
# ============================================================================
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    topic: str
    search_queries: list[str]
    findings: Annotated[list[dict], operator.add]
    analysis: str
    report: str
    quality_score: float
    quality_feedback: str
    iteration: int


# State for individual search tasks (used with Send API)
class SearchTaskState(TypedDict):
    search_query: str
    findings: Annotated[list[dict], operator.add]

---
## 🤖 Part 3: Agent Nodes

Each node below is a single step of the research pipeline: planning, searching, synthesizing, and writing.

### 3.1 🗂️ `supervisor`

Plans the research by asking the LLM to generate exactly three targeted search queries covering different angles of the topic.

In [ ]:
# ============================================================================
# SUPERVISOR: Plans Research by Generating Search Queries
# ============================================================================
def supervisor(state: ResearchState) -> dict:
    """Plans research by generating targeted search queries."""
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a research supervisor. Given a topic, generate exactly 3 "
                    "specific search queries that will cover different angles of the topic. "
                    "Return ONLY a JSON array of strings. No markdown formatting."
                )
            ),
            HumanMessage(content=f"Research topic: {state['topic']}"),
        ]
    )

    try:
        queries = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback: split by newlines
        queries = [
            f"{state['topic']} overview",
            f"{state['topic']} latest developments",
            f"{state['topic']} practical applications",
        ]

    return {
        "search_queries": queries[:3],
        "messages": [
            AIMessage(
                content=f"[SUPERVISOR]: Planned {len(queries)} research queries: {queries}",
                name="supervisor",
            )
        ],
    }

### 3.2 🔎 `search_agent`

Executes a single search query and returns 2-3 findings. This node is designed to be launched **in parallel** — one instance per query — via the `Send` API.

In [ ]:
# ============================================================================
# SEARCH AGENT: Executes One Search Query (Runs in Parallel via Send API)
# ============================================================================
def search_agent(state: SearchTaskState) -> dict:
    """
    Executes one search query and returns findings.
    Each instance runs in parallel via the Send API.
    """
    query = state["search_query"]
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a web research agent. For the given search query, "
                    "provide 2-3 key findings. Each finding should have a 'title' "
                    "and 'detail' field. Return a JSON array. No markdown."
                )
            ),
            HumanMessage(content=f"Search query: {query}"),
        ]
    )

    try:
        results = json.loads(response.content)
    except json.JSONDecodeError:
        results = [{"title": query, "detail": response.content}]

    # Tag each finding with the query it came from
    for r in results:
        r["source_query"] = query

    return {"findings": results}

### 3.3 🚀 `dispatch_searches`

The fan-out edge: dynamically creates one `Send("search_agent", ...)` per planned query, so all searches run concurrently instead of sequentially.

In [ ]:
# ============================================================================
# DISPATCH SEARCHES: Dynamic Fan-Out via the Send API
# ============================================================================
def dispatch_searches(state: ResearchState) -> list[Send]:
    """Dynamically create parallel search tasks using Send API."""
    return [
        Send("search_agent", {"search_query": query, "findings": []})
        for query in state["search_queries"]
    ]

### 3.4 🧩 `analyst`

Reads every finding accumulated on the shared blackboard (`state["findings"]`) and synthesizes them into themes, gaps/contradictions, and key insights.

In [ ]:
# ============================================================================
# ANALYST: Synthesizes Findings from the Shared Blackboard
# ============================================================================
def analyst(state: ResearchState) -> dict:
    """Reads all findings from the blackboard and synthesizes."""
    findings_text = json.dumps(state["findings"], indent=2)
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a research analyst. Synthesize the collected findings into "
                    "a clear analysis. Identify:\n"
                    "1. Key themes across all findings\n"
                    "2. Any contradictions or gaps\n"
                    "3. The most important insights\n\n"
                    "Write 2-3 paragraphs."
                )
            ),
            HumanMessage(
                content=(
                    f"Research topic: {state['topic']}\n\n"
                    f"Collected findings:\n{findings_text}"
                )
            ),
        ]
    )
    return {
        "analysis": response.content,
        "messages": [
            AIMessage(content=f"[ANALYST]: {response.content}", name="analyst")
        ],
    }

### 3.5 ✍️ `report_writer`

Drafts a structured report (executive summary, key findings, analysis, recommendations) using the higher-temperature `creative_llm`. On revision passes it also folds in the quality checker's feedback.

In [ ]:
# ============================================================================
# REPORT WRITER: Drafts (or Revises) the Structured Research Report
# ============================================================================
def report_writer(state: ResearchState) -> dict:
    """Writes a structured research report from the analysis."""
    # Include quality feedback if this is a revision
    revision_note = ""
    if state["iteration"] > 0 and state.get("quality_feedback"):
        revision_note = (
            f"\n\nIMPORTANT — This is revision #{state['iteration']}. "
            f"Address this feedback: {state['quality_feedback']}"
        )

    response = creative_llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a report writer. Produce a well-structured research report "
                    "with these sections:\n"
                    "1. Executive Summary (2-3 sentences)\n"
                    "2. Key Findings (bullet points)\n"
                    "3. Analysis (1-2 paragraphs)\n"
                    "4. Recommendations (3 actionable items)\n\n"
                    "Use markdown formatting. Be specific and actionable."
                    f"{revision_note}"
                )
            ),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n\n"
                    f"Analysis:\n{state['analysis']}\n\n"
                    f"Raw findings:\n{json.dumps(state['findings'][:6], indent=2)}"
                )
            ),
        ]
    )
    return {
        "report": response.content,
        "messages": [
            AIMessage(
                content=f"[REPORT WRITER]: Report {'revised' if state['iteration'] > 0 else 'drafted'}.",
                name="report_writer",
            )
        ],
    }

---
## 🔍 Part 4: Quality Review Loop

The report isn't automatically final — a `quality_checker` node scores it with structured output, and a `quality_gate` conditional edge decides whether to approve it or send it back to `report_writer` for another pass. A hard cap of 2 iterations prevents infinite loops.

### Key Insight:
> Iterative refinement loops need a forced exit condition. Here, both `quality_checker` (force-approve) and `quality_gate` (route to `end`) independently check `iteration >= 2`, so the loop always terminates even if the LLM never scores the report as approved.

### 4.1 📊 `QualityReview` and `quality_checker`

`QualityReview` is the Pydantic schema used with `.with_structured_output()` so the reviewer LLM returns a typed score, feedback string, and approval flag instead of free text.

In [ ]:
# ============================================================================
# QUALITY REVIEW SCHEMA: Structured Output for the Reviewer LLM
# ============================================================================
class QualityReview(BaseModel):
    score: float = Field(description="Quality score from 0.0 to 1.0")
    feedback: str = Field(description="Specific feedback for improvement")
    approved: bool = Field(description="Whether the report meets quality standards")

In [ ]:
# ============================================================================
# QUALITY CHECKER: Scores the Report and Decides Approve vs. Revise
# ============================================================================
def quality_checker(state: ResearchState) -> dict:
    """Reviews the report and either approves or sends back for revision."""
    review_llm = llm.with_structured_output(QualityReview)
    review = review_llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a quality reviewer. Score the report on:\n"
                    "- Completeness: Does it cover the topic well?\n"
                    "- Clarity: Is it well-written and easy to understand?\n"
                    "- Actionability: Are recommendations specific?\n\n"
                    "Score from 0.0 to 1.0. Approve if score >= 0.7.\n"
                    "If this is iteration 2 or higher, be more lenient."
                )
            ),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Iteration: {state['iteration']}\n\n"
                    f"Report:\n{state['report']}"
                )
            ),
        ]
    )

    # Force approve after 2 iterations to prevent infinite loops
    approved = review.approved or state["iteration"] >= 2

    return {
        "quality_score": review.score,
        "quality_feedback": review.feedback,
        "iteration": state["iteration"] + 1,
        "messages": [
            AIMessage(
                content=(
                    f"[QUALITY CHECK]: Score {review.score:.1f} — "
                    f"{'APPROVED' if approved else 'REVISION NEEDED'}: {review.feedback}"
                ),
                name="quality_checker",
            )
        ],
    }

### 4.2 🚦 `quality_gate`

The conditional-edge routing function: sends the graph back to `report_writer` if the score is too low (and the iteration cap hasn't been hit), otherwise routes to `END`.

In [ ]:
# ============================================================================
# QUALITY GATE: Routes Back to the Writer or Ends the Graph
# ============================================================================
def quality_gate(state: ResearchState) -> Literal["report_writer", "end"]:
    """Route back to writer if quality is insufficient."""
    if state["quality_score"] >= 0.7 or state["iteration"] >= 2:
        return "end"
    return "report_writer"

---
## 🏗️ Part 5: Assembling the Graph

`create_research_system` wires every node above into one `StateGraph`: `supervisor` fans out to parallel `search_agent`s, which fan back in to `analyst` → `report_writer` → `quality_checker`, with a conditional loop back to `report_writer` until the quality gate is satisfied.

In [ ]:
# ============================================================================
# CREATE RESEARCH SYSTEM: Build and Compile the Multi-Agent Graph
# ============================================================================
def create_research_system():
    """
    Builds the multi-agent research system graph.

    Flow:
    1. Supervisor plans search queries
    2. Search agents run in parallel (Send API)
    3. Analyst synthesizes findings
    4. Report writer produces report
    5. Quality checker reviews — loops back if needed
    """
    graph = StateGraph(ResearchState)

    # Add all nodes
    graph.add_node("supervisor", supervisor)
    graph.add_node("search_agent", search_agent)
    graph.add_node("analyst", analyst)
    graph.add_node("report_writer", report_writer)
    graph.add_node("quality_checker", quality_checker)

    # Edges
    graph.add_edge(START, "supervisor")

    # Supervisor → parallel search agents (dynamic fan-out)
    graph.add_conditional_edges("supervisor", dispatch_searches, ["search_agent"])

    # All search agents → analyst (fan-in)
    graph.add_edge("search_agent", "analyst")

    # Analyst → report writer
    graph.add_edge("analyst", "report_writer")

    # Report writer → quality checker
    graph.add_edge("report_writer", "quality_checker")

    # Quality gate: approve or revise
    graph.add_conditional_edges(
        "quality_checker", quality_gate, {"report_writer": "report_writer", "end": END}
    )

    return graph.compile()

---
## 🎬 Part 6: Demos

Three demo functions exercise the system at different levels: a single search agent in isolation, the full pipeline with step-by-step streaming, and the full pipeline running end-to-end with a final printed report.

### 6.1 🧪 `demo_individual_search`

Calls `search_agent` directly (outside the graph) to sanity-check a single search step in isolation.

In [ ]:
# ============================================================================
# DEMO: Individual Search Agent (Isolated Test)
# ============================================================================
def demo_individual_search():
    """Demo just the search agent for testing."""
    print("Individual Search Agent Test:\n")

    # Test the search agent directly
    result = search_agent(
        {"search_query": "LangGraph multi-agent patterns", "findings": []}
    )

    print(f"Findings from search:")
    for f in result["findings"]:
        print(f"  - {f.get('title', 'N/A')}: {f.get('detail', 'N/A')[:80]}...")

### 6.2 📡 `demo_research_with_streaming`

Runs the full graph with `stream_mode="updates"` so each node's output is printed as it completes, and saves a Mermaid PNG of the compiled graph to `research_graph.png`.

In [ ]:
# ============================================================================
# DEMO: Full Research System with Step-by-Step Streaming
# ============================================================================
def demo_research_with_streaming():
    """Run the research system with step-by-step streaming output."""
    system = create_research_system()
    graph = system.get_graph()
    png_data = graph.draw_mermaid_png()
    with open("research_graph.png", "wb") as f:
        f.write(png_data)

    topic = "Best practices for building multi-agent AI systems"
    print(f"Streaming Research: {topic}\n")

    initial_state = {
        "messages": [],
        "topic": topic,
        "search_queries": [],
        "findings": [],
        "analysis": "",
        "report": "",
        "quality_score": 0.0,
        "quality_feedback": "",
        "iteration": 0,
    }

    # Stream updates to see each step as it happens
    for step in system.stream(initial_state, stream_mode="updates"):
        for node_name, update in step.items():
            print(f"[{node_name}] completed")

            # Show interesting state changes
            if "search_queries" in update and update["search_queries"]:
                print(f"  Planned queries: {update['search_queries']}")
            if "findings" in update and update["findings"]:
                print(f"  Found {len(update['findings'])} results")
            if "quality_score" in update:
                print(f"  Quality score: {update['quality_score']:.1f}")
            if "report" in update and update["report"]:
                print(f"  Report length: {len(update['report'])} chars")

        print()

### 6.3 📄 `demo_full_research`

Runs the full pipeline non-streaming with `.invoke()`, then prints the agent activity log, final stats (findings count, quality score, iterations), and the finished report.

In [ ]:
# ============================================================================
# DEMO: Full Research Pipeline (Non-Streaming, with Final Report)
# ============================================================================
def demo_full_research():
    """Run the complete research pipeline."""
    system = create_research_system()

    print("Multi-Agent Research System Demo")
    print("=" * 60)

    topic = "The impact of AI agents on software development in 2026"
    print(f"Topic: {topic}\n")

    result = system.invoke(
        {
            "messages": [],
            "topic": topic,
            "search_queries": [],
            "findings": [],
            "analysis": "",
            "report": "",
            "quality_score": 0.0,
            "quality_feedback": "",
            "iteration": 0,
        }
    )

    # Print the conversation trace
    print("Agent Activity Log:")
    print("-" * 40)
    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            print(f"  {msg.content[:120]}...")
    print()

    # Print final stats
    print(f"Total findings collected: {len(result['findings'])}")
    print(f"Quality score: {result['quality_score']:.1f}")
    print(f"Iterations: {result['iteration']}")
    print()

    # Print the final report
    print("=" * 60)
    print("FINAL RESEARCH REPORT")
    print("=" * 60)
    print(result["report"])

---
## ▶️ Run

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is. Uncomment a line to run that demo instead of the default full-pipeline demo.

In [ ]:
# ============================================================================
# RUN: Execute a Demo (Uncomment the One You Want)
# ============================================================================
if __name__ == "__main__":
    # Run the individual search agent demo
    # demo_individual_search()
    # print("\n" + "=" * 50 + "\n")

    # Run the full research system demo with streaming output
    # demo_research_with_streaming()
    # print("\n" + "=" * 50 + "\n")

    # Run the full research system demo without streaming output
    demo_full_research()

---
## 📝 Summary

In this notebook, we combined every Section 4 Multi-Agent pattern into one working research pipeline.

### 1. Patterns Applied
- **Supervisor architecture**: `supervisor` decomposes a broad topic into 3 targeted search queries
- **Parallel execution (`Send` API)**: `dispatch_searches` fans out one `search_agent` per query, running concurrently
- **Shared state / blackboard**: `findings` uses `Annotated[list[dict], operator.add]` so parallel agents safely accumulate into one list
- **Iterative refinement loop**: `quality_checker` + `quality_gate` route the report back to `report_writer` until it scores well or hits the iteration cap

### 2. Components Defined
- State: `ResearchState`, `SearchTaskState`
- Nodes: `supervisor`, `search_agent`, `analyst`, `report_writer`, `quality_checker`
- Routing: `dispatch_searches` (fan-out), `quality_gate` (revise-or-end)
- Schema: `QualityReview` (structured output for scoring)
- Graph: `create_research_system()`
- Demos: `demo_individual_search()`, `demo_research_with_streaming()`, `demo_full_research()`

### Next Steps
- This is the last notebook in `04_Multi_Agent_Systems/` — continue on to **`05_Production_and_Operations/`** to learn how to take agentic systems like this one to production (observability, evaluation, deployment, and operational concerns)
- As a stretch exercise, try adding a second parallel branch (e.g., a fact-checking agent) alongside `search_agent` to see how the blackboard pattern scales to more concurrent contributors